# Peptides-struct seed 2: canonical scores + carriage (5 checkpoints)

Use an **A100 + high-RAM** runtime, grant this notebook access to the Colab secret **`dissertation_key`**, and choose **Runtime → Run all**. This notebook verifies the pinned `expansion/carriage_experiments` checkout, verifies/extracts the uploaded corpus automatically, then runs dense, 1-hop, 1-hop+VNode, 2-hop, and 2-hop+VNode for training seed 2 in sequence.

Every graph shard and consolidated cache is written directly to Drive. Rerunning validates and skips completed checkpoints, and resumes partial checkpoints. Only set `RECLAIM_SETUP_LOCK = True` after confirming the setup runtime has died; reset it immediately afterward. After all six notebooks reach 5/5, set `MODE = "finalize"` in any one notebook and rerun its final cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
MODE = "run"  # @param ["run", "setup", "status", "finalize"]
DATASET = "struct"
TRAIN_SEED = 2  # @param {type:"integer"}
DRIVE_FOLDER = "/content/drive/MyDrive/graph_specialisation_metrics/multi_seed_models/peptides_func_struct_checkpoints"
GRAPHS_PER_BATCH = 0  # @param {type:"integer"}
RECLAIM_SETUP_LOCK = False  # @param {type:"boolean"}
RECLAIM_WORKER_INDEX = -1  # @param {type:"integer"}
STRICT_AUDITS = False  # @param {type:"boolean"}
REPO_BRANCH = "expansion/carriage_experiments"
REPO_REVISION = "a67379fecd6ff7da5eb8a1ce741a9ecbdd596066"
REPO_DIR = "/content/Graph-Specialisation-and-Metrics"
GITHUB_SECRET = "dissertation_key"

In [ ]:
import base64
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

repo = Path(REPO_DIR)
public_url = 'https:' + '//github.com/joshgreenwa/Graph-Specialisation-and-Metrics.git'
try:
    token = userdata.get(GITHUB_SECRET)
except Exception as error:
    raise RuntimeError(
        f'Grant this notebook access to the Colab secret {GITHUB_SECRET!r} and rerun.'
    ) from error
token = str(token).strip() if token else ''
if not token:
    raise RuntimeError(
        f'Colab secret {GITHUB_SECRET!r} is missing or empty. Add it under Secrets and rerun.'
    )
if repo.exists() and not (repo / '.git').is_dir():
    print(f'[repo] removing incomplete checkout at {repo}')
    shutil.rmtree(repo)
credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ.copy()
git_env.update({
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'Authorization: Basic {credential}',
})
remote_ref = f'refs/remotes/origin/{REPO_BRANCH}'
try:
    if not (repo / '.git').is_dir():
        subprocess.run(
            ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', '--no-tags', public_url, str(repo)],
            check=True,
            env=git_env,
        )
    else:
        remotes = subprocess.check_output(
            ['git', '-C', str(repo), 'remote'], text=True
        ).split()
        remote_action = 'set-url' if 'origin' in remotes else 'add'
        subprocess.run(
            ['git', '-C', str(repo), 'remote', remote_action, 'origin', public_url],
            check=True,
        )
    subprocess.run(
        ['git', '-C', str(repo), 'config', '--local', '--unset-all', 'http.https://github.com/.extraheader'],
        check=False,
    )
    subprocess.run(
        ['git', '-C', str(repo), 'fetch', '--no-tags', 'origin', f'+refs/heads/{REPO_BRANCH}:{remote_ref}'],
        check=True,
        env=git_env,
    )
    subprocess.run(
        ['git', '-C', str(repo), 'cat-file', '-e', f'{REPO_REVISION}^{{commit}}'],
        check=True,
    )
    ancestry = subprocess.run(
        ['git', '-C', str(repo), 'merge-base', '--is-ancestor', REPO_REVISION, remote_ref],
        check=False,
    )
    if ancestry.returncode != 0:
        raise RuntimeError(f'Pinned revision {REPO_REVISION} is not on {REPO_BRANCH}.')
    subprocess.run(
        ['git', '-C', str(repo), 'checkout', '--force', '-B', REPO_BRANCH, REPO_REVISION],
        check=True,
    )
finally:
    git_env.pop('GIT_CONFIG_VALUE_0', None)
    token = None
    credential = None
branch = subprocess.check_output(
    ['git', '-C', str(repo), 'branch', '--show-current'], text=True
).strip()
head = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
remote_url = subprocess.check_output(
    ['git', '-C', str(repo), 'remote', 'get-url', 'origin'], text=True
).strip()
if branch != REPO_BRANCH or head != REPO_REVISION or remote_url != public_url:
    raise RuntimeError(
        f'Repository verification failed: branch={branch!r}, head={head!r}, remote={remote_url!r}.'
    )
python_paths = [str(repo), str(repo / 'src')]
sys.path[:] = [path for path in sys.path if path not in python_paths]
sys.path[:0] = python_paths
for name in tuple(sys.modules):
    if name in {
        'experiments.methodology.peptides_func_struct_canonical_colab',
        'experiments.methodology.zinc_qm9_canonical_colab_worker',
    } or name == 'graph_specialisation_metrics' or name.startswith('graph_specialisation_metrics.'):
        del sys.modules[name]
methodology_package = sys.modules.get('experiments.methodology')
if methodology_package is not None:
    vars(methodology_package).pop('peptides_func_struct_canonical_colab', None)
    vars(methodology_package).pop('zinc_qm9_canonical_colab_worker', None)
importlib.invalidate_caches()
from experiments.methodology import peptides_func_struct_canonical_colab as controller_module
controller_path = Path(controller_module.__file__).resolve()
if not controller_path.is_relative_to(repo.resolve()):
    raise RuntimeError(f'Imported stale controller from {controller_path}.')
print(f'[repo] verified branch={branch} head={head} controller={controller_path}')

In [ ]:
from experiments.methodology.peptides_func_struct_canonical_colab import run_frontend

result = run_frontend(
    mode=MODE,
    dataset=DATASET,
    train_seed=TRAIN_SEED,
    drive_folder=DRIVE_FOLDER,
    graphs_per_batch=GRAPHS_PER_BATCH or None,
    accelerator='cuda:0',
    strict_audits=STRICT_AUDITS,
    reclaim_setup_lock=RECLAIM_SETUP_LOCK,
    reclaim_worker_index=RECLAIM_WORKER_INDEX,
)